# Evaluation - Finding and Fixing Bugs with NAT Eval

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

# Verify it loaded
print("API key set:", "Yes" if os.getenv('NVIDIA_API_KEY') else "No")

API key set: Yes


In [2]:
%%capture
# Install the climate analyzer package
!cd climate_analyzer && pip install -e . && cd ..

## Evaluation Dataset
An evaluation dataset consists of questions paired with ground truth answers. The agent's responses are compared against these known-correct answers to calculate accuracy.

In [ ]:
# %load climate_analyzer/data/simple_eval.json
[
  {
    "id": "austria_1980",
    "question": "What was the average temperature in Austria in 1980? Please provide the numerical value.",
    "answer": "The average temperature in Austria in 1980 was 6.80\u00b0C"
  }
]


### Verify Ground Truth

In [4]:
!grep "^[^,]*,1980,[^,]*,Austria" ../resources/climate_data/temperature_annual.csv

AU000005010,1980,AU,Austria,48.05,14.1331,KREMSMUENSTER,7.994166666666666
AUXLT782426,1980,AU,Austria,47.0,15.4333,GRAZ_THALERHOF,7.631666666666667
AUXLT891651,1980,AU,Austria,47.383,13.456,RADSTADT,4.766666666666667


<div style="background-color: #f5f5f5; border: 1px solid #ddd; padding: 15px; border-radius: 5px; margin: 15px 0; font-family: monospace;">
<strong>Raw Data for Austria 1980:</strong>
<pre style="margin: 10px 0; white-space: pre-wrap;">
Austria,1980,01,7.994166666666666
Austria,1980,02,7.631666666666667
Austria,1980,03,4.766666666666667
</pre>
</div>

In [5]:
# Calculate the average to confirm our ground truth:
temps = [7.994166666666666, 7.631666666666667, 4.766666666666667]
average = sum(temps) / len(temps)
print(f"\nAverage temperature for Austria in 1980: {average:.2f}°C")


Average temperature for Austria in 1980: 6.80°C


Add an eval section to your NAT config to define your test dataset and metrics:
<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📋 Evaluation Configuration</h4>
<pre style="background-color: #f5f5f5; padding: 10px; border-radius: 3px; margin: 10px 0;">
eval:
  eval_dataset_file_path: data/simple_eval.json  # Test questions + answers
  eval_name: simple_test                          # Name for this eval run
  eval_output_folder_path: .tmp/nat/climate_analyzer/eval  # Where to save results
  eval_metrics:
    - _type: answer_accuracy                      # Metric: compare answers
      model_name: meta/llama-3.1-70b-instruct    # LLM judges accuracy

In [ ]:
# %load climate_analyzer/src/climate_analyzer/configs/eval_config.yml
llms:
  climate_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    base_url: $NVIDIA_BASE_URL 
    api_key: $NVIDIA_API_KEY
    temperature: 0.7
    top_p: 0.95
    max_tokens: 2048
  
  calculator_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    base_url: $NVIDIA_BASE_URL
    api_key: $NVIDIA_API_KEY
    temperature: 0.0
    max_tokens: 1024

functions:
  list_countries:
    _type: climate_analyzer/list_countries
    description: "List all available countries in the dataset"
    
  calculate_statistics:
    _type: climate_analyzer/calculate_statistics
    description: "Calculate temperature statistics globally or for a specific country"
  
  filter_by_country:
    _type: climate_analyzer/filter_by_country
    description: "Get information about climate data for a specific country"
  
  find_extreme_years:
    _type: climate_analyzer/find_extreme_years
    description: "Find the warmest or coldest years in the dataset"
  
  create_visualization:
    _type: climate_analyzer/create_visualization
    description: "Create visualizations including automatic top 5 countries by warming trend (country_comparison plot)"

  station_statistics:
    _type: climate_analyzer/station_statistics
    description: "Get statistics on climate stations used in the data"
  
  calculator_agent:
    _type: climate_analyzer/calculator_agent
    description: "Perform complex mathematical calculations for climate data analysis"

workflow:
  _type: react_agent
  tool_names:
    - list_countries
    - calculate_statistics
    - filter_by_country
    - find_extreme_years
    - create_visualization
    - station_statistics
    - calculator_agent
  llm_name: climate_llm
  max_iterations: 5
  parse_agent_response_max_retries: 2
  max_tool_calls: 30

# Evaluation configuration
eval:
  general:
    output:
      dir: ./.tmp/nat/climate_analyzer/eval/simple_test/
      cleanup: false  # Keep results for inspection
    dataset:
      _type: json
      file_path: data/simple_eval.json

  evaluators:
    # Check if the answer is accurate
    answer_accuracy:
      _type: ragas
      metric: AnswerAccuracy
      llm_name: climate_llm


## Run Evaluation

In [7]:
!cd climate_analyzer && nat eval --config_file src/climate_analyzer/configs/eval_config.yml

2026-04-18 07:18:53 - INFO     - nat.eval.evaluate:448 - Starting evaluation run with config file: src/climate_analyzer/configs/eval_config.yml
2026-04-18 07:18:55 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
2026-04-18 07:18:55 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
/usr/local/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:171: UserWarning: http://jupyter-api-proxy.internal.dlai/rev-proxy/nvidia does not end in /v1, you may have inference and listing issues. This check will be deprecated in the next release. Please ensure /v1 is appended to the provided URL
  warnings.warn(
/usr/local/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:171: UserWarning: http://jupyter-api-proxy.internal.dlai/rev-proxy/nvidia does not end in /v1, you may have inference and listing issues. This check will be deprecated in the next release. Please 

## Check Results

### Score Summary

In [8]:
import json

with open('climate_analyzer/.tmp/nat/climate_analyzer/eval/simple_test/answer_accuracy_output.json', 'r') as f:
    answer_accuracy_data = json.load(f)

In [9]:
print(f"📊 Evaluation Results")
print(f"=" * 50)
print(f"Average Score: {answer_accuracy_data['average_score']} / 1.0")
print()

for item in answer_accuracy_data['eval_output_items']:
    r = item['reasoning']
    print(f"❓ {r['user_input']}")
    print(f"✅ Expected: {r['reference']}")
    print(f"❌ Got: {r['response']}")
    print(f"📈 Score: {item['score']}")
    print()

📊 Evaluation Results
Average Score: 0.0 / 1.0

❓ What was the average temperature in Austria in 1980? Please provide the numerical value.
✅ Expected: The average temperature in Austria in 1980 was 6.80°C
❌ Got: 8.08
📈 Score: 0.0



In [10]:
# Extract the reasoning steps
item = answer_accuracy_data['eval_output_items'][0]
contexts = item['reasoning']['retrieved_contexts']

print("🤖 AGENT'S DECISION PROCESS")
print("=" * 60)
print(f"Question: {item['reasoning']['user_input']}")
print(f"Expected: {item['reasoning']['reference']}")
print("=" * 60)
print()

# Parse each step
for i, context in enumerate(contexts):
    if context.startswith('**Step'):
        # Extract step number and content
        lines = context.strip().split('\n')
        step_header = lines[0]
        
        print(f"{step_header}")
        
        # Look for Thought
        if 'Thought:' in context:
            thought_start = context.find('Thought:') + 8
            thought_end = context.find('\n\nAction:') if '\n\nAction:' in context else len(context)
            thought = context[thought_start:thought_end].strip()
            print(f"💭 Thought: {thought}")
        
        # Look for Action (tool call)
        if 'Action:' in context and 'Action Input:' in context:
            action_start = context.find('Action:') + 7
            action_end = context.find('\nAction Input:')
            action = context[action_start:action_end].strip()
            
            input_start = context.find('Action Input:') + 13
            input_end = context.find('\n\n', input_start) if '\n\n' in context[input_start:] else len(context)
            action_input = context[input_start:input_end].strip()
            
            print(f"🛠️  Tool: {action}")
            print(f"📥 Input: {action_input}")
        
        # Look for tool response (usually JSON)
        if i + 1 < len(contexts) and contexts[i + 1].startswith('{'):
            print(f"📤 Response: {contexts[i + 1][:100]}..." if len(contexts[i + 1]) > 100 else f"📤 Response: {contexts[i + 1]}")
        
        # Look for Final Answer
        if 'Final Answer:' in context:
            answer_start = context.find('Final Answer:') + 13
            final_answer = context[answer_start:].strip()
            print(f"✅ Final Answer: {final_answer}")
        
        print()

print("\n" + "=" * 60)
print(f"❌ Actual answer given: {item['reasoning']['response']}")
print(f"📊 Score: {item['score']}")

🤖 AGENT'S DECISION PROCESS
Question: What was the average temperature in Austria in 1980? Please provide the numerical value.
Expected: The average temperature in Austria in 1980 was 6.80°C

**Step 0**
💭 Thought: To find the average temperature in Austria in 1980, we need to calculate the temperature statistics for Austria and then filter the results to get the average temperature for 1980.
🛠️  Tool: calculate_statistics
📥 Input: {"country": "Austria"}

**Step 1**

**Step 2**
💭 Thought: The provided temperature statistics for Austria include the mean temperature, but it's for the entire period of 1950-2025. To find the average temperature in Austria in 1980, we would need more specific data or a different approach. However, we can use the calculator_agent tool to find the average temperature in 1980, assuming we have the monthly temperatures for that year.
🛠️  Tool: calculator_agent
📥 Input: {"question": "What is the average temperature in Austria in 1980 given the monthly temperatures

## Test the Fix

In [11]:
# Run evaluation with the fixed tool
!cd climate_analyzer && nat eval --config_file src/climate_analyzer/configs/eval_config_fixed.yml

2026-04-18 07:20:11 - INFO     - nat.eval.evaluate:448 - Starting evaluation run with config file: src/climate_analyzer/configs/eval_config_fixed.yml
2026-04-18 07:20:13 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
2026-04-18 07:20:13 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
2026-04-18 07:20:13 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
/usr/local/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:171: UserWarning: http://jupyter-api-proxy.internal.dlai/rev-proxy/nvidia does not end in /v1, you may have inference and listing issues. This check will be deprecated in the next release. Please ensure /v1 is appended to the provided URL
  warnings.warn(
/usr/local/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:171: UserWarning: http://jupyter-api-proxy.internal.dlai/rev-proxy/nv

## Verify Results
Now that the logic has been updated, check the results again to see if the score improved: 

In [12]:
import json

with open('climate_analyzer/.tmp/nat/climate_analyzer/eval/fixed_test/answer_accuracy_output.json', 'r') as f:
    answer_accuracy_data = json.load(f)

In [13]:
print(f"📊 Evaluation Results")
print(f"=" * 50)
print(f"Average Score: {answer_accuracy_data['average_score']} / 1.0")
print()

for item in answer_accuracy_data['eval_output_items']:
    r = item['reasoning']
    print(f"❓ {r['user_input']}")
    print(f"✅ Expected: {r['reference']}")
    print(f"❌ Got: {r['response']}")
    print(f"📈 Score: {item['score']}")
    print()

📊 Evaluation Results
Average Score: 1.0 / 1.0

❓ What was the average temperature in Austria in 1980? Please provide the numerical value.
✅ Expected: The average temperature in Austria in 1980 was 6.80°C
❌ Got: 6.8
📈 Score: 1.0

